In [16]:
import numpy as np

# =====================================================================
# UPDATED TIMELINE PARAMETERS
# =====================================================================
T = 30.0       # Total time horizon = 30 min
nsteps = 30    # N = 30 discrete time steps (resulting in dt = 1.0 min)

# =====================================================================
# TARGET SETPOINTS (SP) FROM TABLE 2
# =====================================================================
SP = {
    'CV': [1.00 for _ in range(nsteps)],
    'Ln': [15.00 for _ in range(nsteps)]
}

# =====================================================================
# UPDATED ACTION SPACE (Incremental control bounds)
# =====================================================================
action_space = {
    'low': np.array([-1.0]),   # Minimum delta T step allowed per minute
    'high': np.array([1.0])    # Maximum delta T step allowed per minute
}

# =====================================================================
# OBSERVATION SPACE (State Bounds from Table 2 + Setpoints)
# =====================================================================
# Order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
observation_space = {
    'low' : np.array([0.0, 0.0, 0.0, 0.0, 0.00, 0.00, 0.00, 0.00, 0.00]),
    'high' : np.array([1.0e20, 1.0e20, 1.0e20, 1.0e20, 0.50, 2.00, 20.00, 2.00, 20.00])  
}

# =====================================================================
# ENVIRONMENT PARAMETERS SETUP
# =====================================================================
env_params = {
    'N': nsteps, 
    'tsim': T, 
    'SP': SP, 
    'o_space': observation_space, 
    'a_space': action_space,
    
    # Initial conditions (x0) including the initial setpoints
    'x0': np.array([1.50e3, 2.30e4, 1.80e6, 2.50e8, 0.16, 1.00, 15.00, 1.00, 15.00]),
    
    'r_scale': {
        'CV': 1e1,
        'Ln': 1e0
    },
    
    'model': 'crystallization', 
    
    'normalise_a': True, 
    'normalise_o': True, 
    'noise': True, 
    'integration_method': 'jax', 
    'noise_percentage': 0.01, 
}

In [21]:
import numpy as np
import pandas as pd
from pcgym import make_env

base_env = make_env(env_params)

inputs_data = []
target_data = []

inputs_data_2 = []
target_data_2 = []

num_episodes = 10  # Try a few episodes to test

for episode in range(num_episodes):
    obs, info = base_env.reset()
    done = False
    truncated = False
    
    while not (done or truncated):
        # =====================================================================
        # FIX: Pull directly from the internal plant state, NOT the 'obs' array
        # =====================================================================
        mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP = obs

        raw_state = base_env.state  
        
        # In pc-gym crystallization, the true state layout is exactly:
        # [mu0, mu1, mu2, mu3, C, CV, Ln]
        mu02 = raw_state[0]
        mu12 = raw_state[1]
        mu22 = raw_state[2]
        mu32 = raw_state[3]
        C2   = raw_state[4]
        CV2  = raw_state[5]
        Ln2  = raw_state[6]
        
        # Grab your setpoints from your original setup dictionary 
        # (or find where your env stores current step setpoints)
        CV_SP = base_env.SP['CV'][base_env.t] if hasattr(base_env, 't') else 1.0
        Ln_SP = base_env.SP['Ln'][base_env.t] if hasattr(base_env, 't') else 15.0

        CV_SP2 = base_env.SP['CV'][base_env.t] if hasattr(base_env, 't') else 1.0
        Ln_SP2 = base_env.SP['Ln'][base_env.t] if hasattr(base_env, 't') else 15.0
        # Save your dataset rows
        inputs_data.append([mu0, mu1, mu2, C, CV, Ln, CV_SP, Ln_SP])
        target_data.append(mu3)

        inputs_data_2.append([mu02, mu12, mu22, C2, CV2, Ln2, CV_SP2, Ln_SP2])
        target_data_2.append(mu32)
        
        # Take a random action to force the states to change!
        random_action = base_env.action_space.sample() 
        obs, reward, done, truncated, info = base_env.step(random_action)

# Convert to DataFrame
X = pd.DataFrame(inputs_data, columns=['mu0', 'mu1', 'mu2', 'C', 'CV', 'Ln', 'CV_SP', 'Ln_SP'])
y = pd.Series(target_data, name='target_mu3')

X2 = pd.DataFrame(inputs_data_2, columns=['mu0', 'mu1', 'mu2', 'C', 'CV', 'Ln', 'CV_SP', 'Ln_SP'])
y2 = pd.Series(target_data_2, name='target_mu3')
# Print the first few rows to verify they are changing
print(X)
print('-'*30)
print(X2)

/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


     mu0  mu1  mu2         C        CV        Ln  CV_SP  Ln_SP
0   -1.0 -1.0 -1.0 -0.360000  0.000000  0.500000    1.0   15.0
1   -1.0 -1.0 -1.0 -0.375896 -0.925969  1.741234    1.0   15.0
2   -1.0 -1.0 -1.0 -0.360378 -1.152412  2.573907    1.0   15.0
3   -1.0 -1.0 -1.0 -0.371331 -1.165084  2.955767    1.0   15.0
4   -1.0 -1.0 -1.0 -0.374966 -1.079882  2.835645    1.0   15.0
..   ...  ...  ...       ...       ...       ...    ...    ...
285 -1.0 -1.0 -1.0 -0.667218 -0.571694  0.550094    1.0   15.0
286 -1.0 -1.0 -1.0 -0.669417 -0.557072  0.519034    1.0   15.0
287 -1.0 -1.0 -1.0 -0.671242 -0.542404  0.502752    1.0   15.0
288 -1.0 -1.0 -1.0 -0.669425 -0.532671  0.445973    1.0   15.0
289 -1.0 -1.0 -1.0 -0.675240 -0.516124  0.432082    1.0   15.0

[290 rows x 8 columns]
------------------------------
               mu0           mu1           mu2         C        CV         Ln  \
0      1500.000000  2.300000e+04  1.800000e+06  0.160000  1.000000  15.000000   
1      1756.236938  4.89130

In [ ]:
import pandas as pd
from pcgym import make_env
from tqdm import trange

# 1. Use your FULL 9-dimensional environment so you can capture the true mu3!
base_env = make_env(env_params)

# Storage lists for your dataset
dataset = []

num_episodes = 1000  # Run multiple batches to collect diverse data

for episode in trange(num_episodes):
    obs, info = base_env.reset()
    done = False
    truncated = False
    
    while not (done or truncated):
        # 2. Extract the physical layout of the current observation
        # obs order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
        
        mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP = obs
        
        # 3. Save your measurable inputs (What the Soft Sensor will see)
        # You can choose to include mu0-mu2, or stick purely to easy measurements like C, CV, Ln.
        current_inputs = [mu0, mu1, mu2, C, CV, Ln, CV_SP, Ln_SP, mu3]
        
        dataset.append(current_inputs)
        
        # 5. Take an exploratory random action to gather diverse state space data
        random_action = base_env.action_space.sample() 
        obs, reward, done, truncated, info = base_env.step(random_action)

# 6. Convert to a Pandas DataFrame for easy Machine Learning training
feature_names = ['mu0', 'mu1', 'mu2', 'C', 'CV', 'Ln', 'CV_SP', 'Ln_SP', 'mu3']
dataset = pd.DataFrame(dataset, columns=feature_names)

print("Dataset ready!")
print("X Shape (Features):", dataset.shape)
print('-'*30)
print(dataset)
dataset.to_csv(f'cryst-{num_episodes}ep.csv', index=False)
print(f'Dataset is saved successfully with {num_episodes} episodes!!!')

Python(13086) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
  0%|          | 2/10000 [00:04<6:08:05,  2.21s/it]


KeyboardInterrupt: 

In [19]:
X.describe()

,mu0,mu1,mu2,C,CV,Ln,CV_SP,Ln_SP
count,1.450000e+03,1.450000e+03,1.450000e+03,1450.000000,1450.000000,1450.000000,1450.0,1450.0
mean,-1.000000e+00,-1.000000e+00,-1.000000e+00,-0.543059,-0.726815,1.264403,0.0,0.5
std,2.485513e-15,3.778432e-14,1.820322e-12,0.118001,0.224714,0.782470,0.0,0.0
min,-1.000000e+00,-1.000000e+00,-1.000000e+00,-0.685700,-1.176730,0.406134,0.0,0.5
25%,-1.000000e+00,-1.000000e+00,-1.000000e+00,-0.653598,-0.833799,0.598948,0.0,0.5
50%,-1.000000e+00,-1.000000e+00,-1.000000e+00,-0.585578,-0.714815,0.978446,0.0,0.5
75%,-1.000000e+00,-1.000000e+00,-1.000000e+00,-0.416189,-0.592146,1.756256,0.0,0.5
max,-1.000000e+00,-1.000000e+00,-1.000000e+00,-0.346657,0.000000,3.045178,0.0,0.5


In [9]:
print(y)

0      -1.0
1      -1.0
2      -1.0
3      -1.0
4      -1.0
       ... 
1445   -1.0
1446   -1.0
1447   -1.0
1448   -1.0
1449   -1.0
Name: target_mu3, Length: 1450, dtype: float64


In [22]:
!pwd

/opt/homebrew/Cellar/python@3.12/3.12.10/Frameworks/Python.framework/Versions/3.12/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/dataset-generation
